In [2]:
import pandas as pd
import json

## Load the data

In [3]:
file_dict = {}
files = {'user_reviews' : '../data/raw/Video_Games.jsonl',
         'meta_data' : '../data/raw/meta_Video_Games.jsonl'}
for name,file in files.items():
    record_list = []
    with open(file, 'r') as fp:
        for line in fp:
            record = json.loads(line.strip())
            record_list.append(record)
            
            
    file_dict[name]=record_list

## User reviews

### Description

In [4]:
user_reviews = pd.DataFrame(file_dict['user_reviews'])
user_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,[],B07DJWBYKP,B07DK1H3H5,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1608186804795,0,True
1,5.0,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,[],B00ZS80PC2,B07SRWRH5D,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1587051114941,1,False
2,5.0,... an order for my kids & they have really en...,This was an order for my kids & they have real...,[],B01FEHJYUU,B07MFMFW34,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,1490877431000,0,True
3,5.0,Great alt to pro controller,"These work great, They use batteries which is ...",[],B07GXJHRVK,B0BCHWZX95,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,1577637634017,0,True
4,5.0,solid product,I would recommend to anyone looking to add jus...,[],B00HUWA45W,B00HUWA45W,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,1427591932000,0,True


This dataset contains 4624615 records and 10 fields as listed below:

In [5]:
user_reviews.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 4624615 entries, 0 to 4624614
Data columns (total 10 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   rating             4624615 non-null  float64
 1   title              4624615 non-null  str    
 2   text               4624615 non-null  str    
 3   images             4624615 non-null  object 
 4   asin               4624615 non-null  str    
 5   parent_asin        4624615 non-null  str    
 6   user_id            4624615 non-null  str    
 7   timestamp          4624615 non-null  int64  
 8   helpful_vote       4624615 non-null  int64  
 9   verified_purchase  4624615 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 322.0+ MB


Each record contains a user review :

In [6]:
file_dict['user_reviews'][0]

{'rating': 4.0,
 'title': 'It’s pretty sexual. Not my fav',
 'text': 'I’m playing on ps5 and it’s interesting.  It’s unique, massive, and has a neat story.  People are freaking out angry about this game.  I don’t think it’s a top 10 game but it’s definitely a good game on ps5 (played at launch).',
 'images': [],
 'asin': 'B07DJWBYKP',
 'parent_asin': 'B07DK1H3H5',
 'user_id': 'AGCI7FAH4GL5FI65HYLKWTMFZ2CQ',
 'timestamp': 1608186804795,
 'helpful_vote': 0,
 'verified_purchase': True}

### Field selection

Field descriptions taken from :[https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews)

|Field|	Type|Explanation| Keep ?|
|-----|----|------------|-------|
|rating|float|	Rating of the product (from 1.0 to 5.0).|No|
|title|	str |	Title of the user review.|Yes|
|text	|str	|Text body of the user review.|Yes|
|images|	list|	Images that users post after they have received the product. Each image has different sizes (small, medium, large), represented by the small_image_url, medium_image_url, and large_image_url respectively.|No|
|asin	|str	|ID of the product.|No|
|parent_asin|	str	|Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta.|Yes|
|user_id	|str|	ID of the reviewer|No|
|timestamp	|int|	Time of the review (unix time)|No|
|verified_purchase	|bool	|User purchase verification|Yes|
|helpful_vote|	int	|Helpful votes of the review|No|


Useful fields:  
- title : needed for search, 
- text : needed for search, 
- parent_asin : needed for tracking product across files , 
- verified_purchase : could only keep verified_purchase=True , 


Less useful fields: 
- rating : can use average_rating from `meta_data`,
- images : many don't contain images, not useful for query search 
- asin : information already given in parent_asin,
- timestamp : time of review might not be useful for query search
- user_id : not useful for query search 
- helpful_vote : most have 0

If we only keep `verified_purchase=True`, we still have enough data points to make our analysis. This would ensure we only use true reviews which can help make the system more accurate.

In [ ]:
user_reviews['verified_purchase'].value_counts()

verified_purchase
True     3982807
False     641808
Name: count, dtype: int64

Only 4% of rows have images, we can remove the column.

In [8]:
round((user_reviews['images'].str.len() > 0).sum()/len(user_reviews['images'])*100,2)

np.float64(4.43)

## Item meta data 

### Description

In [5]:
meta_data = pd.DataFrame(file_dict['meta_data'])
meta_data.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Video Games,Dash 8-300 Professional Add-On,5.0,1,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,[The Dash 8-300 Professional Add-On lets you p...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aerosoft,"[Video Games, PC, Games]",{'Pricing': 'The strikethrough price is the Li...,B000FH0MHO,None,NaN,NaN
1,Video Games,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,None,NaN,NaN
2,Video Games,NBA 2K17 - Early Tip Off Edition - PlayStation 4,4.3,223,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.0,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'NBA 2K17 - Kobe: Haters vs Players...,2K,"[Video Games, PlayStation 4, Games]","{'Release date': 'September 16, 2016', 'Best S...",B00Z9TLVK0,None,NaN,NaN
3,Video Games,Nintendo Selects: The Legend of Zelda Ocarina ...,4.9,22,[Authentic Nintendo Selects: The Legend of Zel...,[],37.42,[{'thumb': 'https://m.media-amazon.com/images/...,[],Amazon Renewed,"[Video Games, Legacy Systems, Nintendo Systems...","{'Best Sellers Rank': {'Video Games': 51019, '...",B07SZJZV88,None,NaN,NaN
4,Video Games,Thrustmaster Elite Fitness Pack for Nintendo Wii,3.0,3,"[Includes (9) Total Accessories, Pedometer, Wi...",[The Thrustmaster Motion Plus Elite Fitness Pa...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],THRUSTMASTER,"[Video Games, Legacy Systems, Nintendo Systems...","{'Release date': 'November 1, 2009', 'Pricing'...",B002WH4ZJG,None,NaN,NaN


This dataset contains 137269 records and 16 fields as listed below:

In [10]:
meta_data.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 137269 entries, 0 to 137268
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    126234 non-null  str    
 1   title            137269 non-null  str    
 2   average_rating   137269 non-null  float64
 3   rating_number    137269 non-null  int64  
 4   features         137269 non-null  object 
 5   description      137269 non-null  object 
 6   price            62008 non-null   object 
 7   images           137269 non-null  object 
 8   videos           137269 non-null  object 
 9   store            132908 non-null  str    
 10  categories       137269 non-null  object 
 11  details          137269 non-null  object 
 12  parent_asin      137269 non-null  str    
 13  bought_together  0 non-null       object 
 14  subtitle         350 non-null     str    
 15  author           262 non-null     object 
dtypes: float64(1), int64(1), object(9), str(5)
memory

Each record contains a information about a product :

In [14]:
file_dict['meta_data'][0]

{'main_category': 'Video Games',
 'title': 'Dash 8-300 Professional Add-On',
 'average_rating': 5.0,
 'rating_number': 1,
 'features': ["Features Dash 8-300 and 8-Q300 ('Q' rollout livery)",
  'Airlines - US Airways, South African Express, Bahamasair, Augsburg Airways, Lufthansa Cityline, British Airways (Union Jack), British European, FlyBe, Intersky, Wideroe, Iberia, Tyrolean, QantasLink, BWIA',
  'Airports include - London City, Frankfurt, Milan and Amsterdam Schipol',
  'Includes PSS PanelConfig and LoadEdit tools'],
 'description': ['The Dash 8-300 Professional Add-On lets you pilot a real commuter special. Fly two versions of the popular Dash 8-300 in a total of 17 different liveries. The Dash 8-300 is one of the most popular short-haul aircraft available and this superbly modelled version from acclaimed aircraft developers PSS is modelled in two versions with a total of 17 different liveries. The package also includes scenery for three European airports, tutorials, tutorial flig

### Field selection

Field descriptions taken from :[https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata)

|Field|	Type|Explanation| Keep ?|
|-----|----|------------|-------|
|main_category	|str|	Main category (i.e., domain) of the product.| No |
|title	|str	|Name of the product.| Yes |
|average_rating|	float|	Rating of the product shown on the product page.| Yes |
|rating_number|	int	|Number of ratings in the product.|Yes |
|features	|list	|Bullet-point format features of the product.|Yes |
|description|	list	|Description of the product.|Yes |
|price	|float|Price in US dollars (at time of crawling).|Yes |
|images|	list|	Images of the product. Each image has different sizes (thumb, large, hi_res). The “variant” field shows the position of image.|No |
|videos|	list|	Videos of the product including title and url.|No |
|store|	str	|Store name of the product.|Yes |
|categories	|list	|Hierarchical categories of the product.|Yes |
|details|	dict	|Product details, including materials, brand, sizes, etc.|Yes |
|parent_asin	|str	|Parent ID of the product.|Yes |
|bought_together	|list	|Recommended bundles from the websites.|No |

The `main_category` field gives a broad description of what type of product we are looking at:

In [11]:
meta_data['main_category'].value_counts()

main_category
Video Games                     81255
Computers                       17235
All Electronics                 14816
Cell Phones & Accessories        3884
Toys & Games                     2733
Software                         1511
Industrial & Scientific          1079
Amazon Home                       737
Home Audio & Theater              443
Tools & Home Improvement          369
Office Products                   295
Sports & Outdoors                 244
Buy a Kindle                      220
Movies & TV                       197
Books                             196
Musical Instruments               154
All Beauty                        126
Camera & Photo                    117
Portable Audio & Accessories      112
Digital Music                     104
Health & Personal Care             95
Automotive                         85
AMAZON FASHION                     54
Pet Supplies                       38
Grocery                            36
Baby                               2

The `categories` field gives more precise information about the products :

In [12]:
meta_data['categories'].value_counts()

categories
[Video Games, PC, Games]                                                                                             17758
[]                                                                                                                   12637
[Video Games, PC, Accessories, Gaming Keyboards]                                                                      4736
[Video Games, PC, Accessories, Headsets]                                                                              3945
[Video Games, PC, Accessories, Gaming Mice]                                                                           3798
                                                                                                                     ...  
[Video Games, FREE One-Day Shipping on Select Video Games with your Discover Card]                                       1
[Video Games, Legacy Systems, PlayStation Systems, PlayStation 3, Accessories, Controllers, Music Controllers]           1
[Vide

Since `categories` and `main_category` give similar information, keeping only 1 seems reasonable. 

Only 21% of rows have videos, and the field is not useful for our query.

In [13]:
round((meta_data['videos'].str.len() > 0).sum()/len(meta_data['images'])*100,2)

np.float64(21.06)

Useful fields: 
- title : needed for search, 
- average_rating : needed for dashboard, 
- rating_number : could be useful for ranking 
- features : needed for search,
- description : needed for search,
- price : useful for search, a lot of nulls
- store : useful for search
- details
- parent_asin


Less useful fields: 
- main_category : not very informative, can just use info from  categories
- images
- videos 
- bought_together : all empty 

## Merging datasets

In [6]:
# Keep only useful fields 
proc_meta_data = meta_data.drop(columns=['main_category','images','videos','bought_together']).rename(columns={'title':'item_title'}) 

# Keep only verified purcahses and useful fields
proc_user_reviews = user_reviews[user_reviews['verified_purchase']==True].drop(columns=['rating','images','asin','timestamp','user_id','helpful_vote','verified_purchase']).rename(columns={'title':'review_title'}) 

In [7]:
merged = pd.merge(proc_meta_data, 
                  proc_user_reviews, 
                  on='parent_asin', 
                  how='inner')

In [23]:
merged.head()


,item_title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin,subtitle,author,review_title,text
0,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,NaN,NaN,Total waste of money and disappointment!,Came missing one disk which makes the game tot...
1,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,NaN,NaN,The package came properly wrapped and the disc...,Phantasmagoria is an jewel in video games hist...
2,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,NaN,NaN,best fmv game is phantasmagoria a puzzle of flesh,i love playing this game its very intense the ...
3,NBA 2K17 - Early Tip Off Edition - PlayStation 4,4.3,223,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.0,2K,"[Video Games, PlayStation 4, Games]","{'Release date': 'September 16, 2016', 'Best S...",B00Z9TLVK0,NaN,NaN,Birthday success,My child love getting this for his birthday.
4,NBA 2K17 - Early Tip Off Edition - PlayStation 4,4.3,223,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.0,2K,"[Video Games, PlayStation 4, Games]","{'Release date': 'September 16, 2016', 'Best S...",B00Z9TLVK0,NaN,NaN,Five Stars,"Great game, my son liked it!"


## Preprocessing

Make everything lowercase, remove non numeric or alphabet elements (takes super long to run)

In [ ]:
import re
def preprocess(text):
    if type(text) == list :
        processed_list = []
        for element in text :
            element = element.lower()
            element = re.sub(r"[^a-z0-9\s]", "", element)
            processed_list.append(element)
        return processed_list
    elif type(text) == str :
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s]", "", text)
        return text
    else:
        return text
    

'hardcover  january 6 2022 '

Make into Langchain document

In [ ]:
from langchain_core.documents import Document

doc_list = []
for record in merged.itertuples(index=True):
    document = Document(
    page_content=f"{preprocess(record.item_title)}, {preprocess(record.review_title)}, {preprocess(record.text)}", metadata={"average_rating": record.average_rating,
                                                                                         "rating_number" : record.rating_number,
                                                                                         "features" : preprocess(record.features),
                                                                                         "description" : preprocess(record.description),
                                                                                         "price" : record.price,
                                                                                         "store" : preprocess(record.store),
                                                                                         "categories" : preprocess(record.categories),
                                                                                         "details" : record.details,
                                                                                         "subtitle" : record.subtitle,
                                                                                         "author" :record.author                                                                                        }
)
    doc_list.append(document)
    

/Users/mailysguedon/miniforge3/envs/575-proj/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


KeyboardInterrupt: 

In [ ]:
doc_list

[Document(metadata={'average_rating': 4.1, 'rating_number': 18, 'features': ['Windows 95'], 'description': [], 'price': None, 'store': 'Sierra', 'categories': ['Video Games', 'PC', 'Games'], 'details': {'Best Sellers Rank': {'Video Games': 137612, 'PC-compatible Games': 6707}, 'Pricing': 'The strikethrough price is the List Price. Savings represents a discount off the List Price.', 'Package Dimensions': '5.6 x 4.9 x 0.9 inches; 6.4 Ounces', 'Type of item': 'CD-ROM', 'Rated': 'Mature', 'Is Discontinued By Manufacturer': 'No', 'Item Weight': '6.4 ounces', 'Manufacturer': 'Sierra', 'Date First Available': 'March 30, 2006'}, 'subtitle': nan, 'author': nan}, page_content='phantasmagoria a puzzle of flesh, total waste of money and disappointment, came missing one disk which makes the game totally unplayable what a waste of money especially from a supposedly respectable brand like sierra give me my money back or send me disk 2'),
 Document(metadata={'average_rating': 4.1, 'rating_number': 18,